In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_dir = "/content/drive/My Drive/DeepLense_GSoC_Data"

class DeepLenseDatasetRGB(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform
        self.class_map = {'no_sub': 0, 'cdm': 1, 'vortex': 2}
    def __len__(self): return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = os.path.join(self.root_dir, row['class'], row['filename'])
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, self.class_map[row['class']]

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomVerticalFlip(p=0.5),   
    transforms.RandomRotation(30),          
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

df = pd.read_csv(os.path.join(base_dir, "metadata.csv"))
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class'])

train_loader = DataLoader(DeepLenseDatasetRGB(train_df, base_dir, transform=train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(DeepLenseDatasetRGB(val_df, base_dir, transform=val_transform), batch_size=32, shuffle=False)

model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 3)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
EPOCHS = 15 

print("Starting Robust Training (Augmentation Enabled)")
for epoch in range(EPOCHS):
    model.train()
    correct = 0
    total = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        progress_bar.set_postfix({'Acc': f"{100 * correct / total:.2f}%"})

save_path = os.path.join(base_dir, "resnet18_robust.pth")
torch.save(model.state_dict(), save_path)
print(f"Robust Model Saved: {save_path}")

Starting Robust Training (Augmentation Enabled)


Epoch 15/15: 100%|██████████| 38/38 [00:04<00:00,  8.72it/s, Acc=71.67%]


Robust Model Saved: /content/drive/My Drive/DeepLense_GSoC_Data/resnet18_robust.pth
